In [1]:
import sys, os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
sys.path.append('../Utilities/')
from audio_utils import AudioHandler
from data_utils import TextHandler, FileHandler
import numpy as np
import json
import torch
from T2C import T2CBert, T2CRoberta
from Manager import Manager

path_data = '/home/thaivanphat/OneDrive/WorkSpace/Data/Speech/Changi/VHF/S2R/'
path_bert = '/home/thaivanphat/OneDrive/WorkSpace/Data/Weights/NER/bert-atc'
path_roberta = '/home/thaivanphat/OneDrive/WorkSpace/Data/Weights/NER/roberta-large'
bert = T2CBert(path_model=path_bert)
roberta = T2CRoberta(path_model=path_roberta)
manager = Manager()

In [2]:
from collections import defaultdict
import Levenshtein
from rapidfuzz.fuzz import token_set_ratio
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, precision_recall_fscore_support

def compute_metrics(labels, preds, average="binary"):
    preds = np.array(preds)
    labels = np.array(labels)

    precision = precision_score(labels, preds, average=average, zero_division=0)
    recall = recall_score(labels, preds, average=average, zero_division=0)
    f1 = f1_score(labels, preds, average=average, zero_division=0)
    accuracy = accuracy_score(labels, preds)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy
    }

def compare_results(gt, predict):
    labels = list(set(gt))
    if len(labels) == 2:
        average = 'binary'
    else:
        average = 'macro'
    refs, preds = np.zeros(len(gt)), np.zeros(len(gt))
    for i in range(len(gt)):
        refs[i] = labels.index(gt[i])
        preds[i] = labels.index(predict[i])
    return compute_metrics(refs, preds, average=average)

def compute_per_class_metrics(gt, pred):
    """
    Returns:
      - per-class precision
      - per-class recall
      - per-class f1
      - support (sample count)
      - accuracy
      - weighted F1
    """
    gt = np.array(gt)
    pred = np.array(pred)

    # Get detailed metrics for each class
    precision, recall, f1, support = precision_recall_fscore_support(
        gt, pred, labels=np.unique(gt), zero_division=0
    )

    accuracy = accuracy_score(gt, pred)

    # Weighted F1: sum(F1_i * support_i) / total_samples
    weighted_f1 = (f1 * support).sum() / support.sum()

    return {
        "precision_per_class": precision,
        "recall_per_class": recall,
        "f1_per_class": f1,
        "support_per_class": support,
        "accuracy": accuracy,
        "weighted_f1": weighted_f1
    }

def score_entities_per_label(gt_entities, pred_entities, threshold=70):
    """
    pred_entities: [(text, label)]
    gt_entities:   [(text, label)]
    threshold: fuzzy match threshold
    """

    # Collect all labels that appear in GT (ground truth)
    labels = sorted(list({lbl for _, lbl in gt_entities}))

    # Per-label TP/FP/FN counters
    TP = defaultdict(int)
    FP = defaultdict(int)
    FN = defaultdict(int)

    # For each label we need separate "used GT indices"
    used_gt = {lbl: set() for lbl in labels}

    # Group GT by label for easier lookup
    gt_by_label = defaultdict(list)
    for i, (g_text, g_label) in enumerate(gt_entities):
        gt_by_label[g_label].append((i, g_text))

    # --- MATCHING ---
    for p_text, p_label in pred_entities:
        matched = False

        # If label is not in GT at all → all predictions are FP
        if p_label not in gt_by_label:
            FP[p_label] += 1
            continue

        for i, g_text in gt_by_label[p_label]:
            if i in used_gt[p_label]:
                continue

            sim = token_set_ratio(p_text, g_text)
            if sim >= threshold:
                TP[p_label] += 1
                used_gt[p_label].add(i)
                matched = True
                break

        if not matched:
            FP[p_label] += 1

    # Compute FN per label
    for lbl in labels:
        FN[lbl] = len(gt_by_label[lbl]) - TP[lbl]

    # --- METRICS ---
    per_label_scores = {}
    total_gt = sum(len(gt_by_label[lbl]) for lbl in labels)
    weighted_f1_sum = 0.0

    for lbl in labels:
        tp = TP[lbl]
        fp = FP[lbl]
        fn = FN[lbl]

        precision = tp / (tp + fp + 1e-9)
        recall    = tp / (tp + fn + 1e-9)
        f1        = 2 * precision * recall / (precision + recall + 1e-9)

        support = len(gt_by_label[lbl])
        weighted_f1_sum += f1 * support

        per_label_scores[lbl] = {
            "precision": precision,
            "recall":    recall,
            "f1":        f1,
            "support":   support,
            "TP": tp, "FP": fp, "FN": fn
        }

    weighted_f1 = weighted_f1_sum / (total_gt + 1e-9)

    return {
        "per_label": per_label_scores,
        "weighted_f1": weighted_f1,
    }

def score_entities(gt_entities, pred_entities, threshold=70):
    """
    pred_entities: [(text, label)]
    gt_entities:   [(text, label)]
    threshold: fuzzy match threshold
    """

    TP = 0
    FP = 0
    FN = 0

    used_gt = set()

    for p_text, p_label in pred_entities:
        matched = False
        for i, (g_text, g_label) in enumerate(gt_entities):
            if i in used_gt:
                continue

            # labels must match
            if p_label != g_label:
                continue

            # fuzzy text similarity
            sim = token_set_ratio(p_text, g_text)
            if sim >= threshold:
                TP += 1
                used_gt.add(i)
                matched = True
                break

        if not matched:
            FP += 1

    FN = len(gt_entities) - TP

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return {
        "entity_precision": precision,
        "entity_recall": recall,
        "entity_f1": f1,
        "TP": TP, "FP": FP, "FN": FN
    }

def extract_callsign(fname):
    cpn = list()
    for f in fname.split('_'):
        if not f.isdigit():
            cpn.append(f)
    return ' '.join(cpn)    

def sort_path(path):
    starts = list()
    for i in range(len(path)):
        prefix = os.path.basename(path[i]).split('_')[0]
        starts.append(int(prefix))
    index = np.argsort(starts)
    path_new = list()
    for i in index:
        path_new.append(path[i])
    return path_new

def seperate_tags(pairs):
    texts, tag = '', list()
    for p in range(len(pairs)):
        text, t = pairs[p]
        words = text.split(' ')
        texts = texts + ' ' + ' '.join(words)
        if t == 'O':
            tag.append(t)
        else:
            tag.append('B-' + t)
            if len(words) > 1:
                for _ in range(1, len(words)):
                    tag.append('I-' + t)
    return texts[1:], tag

def extract_data(path_data):
    path_audios, path_txts, route = list(), list(), dict()
    for fname in sorted(os.listdir(path_data)):
        if fname.endswith('.wav'):
            path_audios.append(os.path.join(path_data, fname))
        if fname.endswith('.txt'):
            path_txts.append(os.path.join(path_data, fname))
        if fname.endswith('.json'):
            route = FileHandler.read_json(os.path.join(path_data, fname))
    return sort_path(path_audios), sort_path(path_txts), route

def extract_NER(path_txts):
    transcripts, speakers, intents, tags, pairs = list(), list(), list(), list(), list()
    for i in range(len(path_txts)):
        lines = FileHandler.read_txt(path_txts[i])
        transcripts.append(lines[0])
        speakers.append(lines[1])
        if lines[2] in ['pilot', 'greet', 'standby']:
            intents.append('other')
        else:
            intents.append(lines[2])
        pair = list()
        for j in range(3, len(lines)):
            cpn = lines[j].split(' - ')
            pair.append(cpn)
        pairs.append(pair)
        text, tag = seperate_tags(pair)
        tags.append(tag)
        
    return transcripts, speakers, intents, tags, pairs

def normalize_transcripts(transcripts):
    spokens = list()
    for i in range(len(transcripts)):
        spoken = TextHandler.remove_punctuation(transcripts[i])
        spokens.append(spoken)
    return spokens

def normalize_text(text):
    MERGE_MAP = {'push back': 'pushback', 'stand by': 'standby', 'take off': 'takeoff'}
    filler_words = ['uh', 'ah', 'erm', 'hmm', 'eh', 'mm', 'um', 'uhh', 'err', 'huh', 'ya', 'yeah', 'ehm', 'uhm']

    # --- Process ---
    words = text.lower().split()
    new_words = []
    i = 0

    while i < len(words):
        w = words[i]

        # Skip fillers
        if w in filler_words:
            i += 1
            continue

        # Replace "niner" with "nine"
        if w == 'niner':
            new_words.append('nine')
            i += 1
            continue

        # Merge known ATC phrases
        if i < len(words) - 1:
            pair = f"{w} {words[i + 1]}"
            if pair in MERGE_MAP:
                new_words.append(MERGE_MAP[pair])
                i += 2
                continue

        # Default
        new_words.append(w)
        i += 1

    return ' '.join(new_words)

def remove_commas(pairs):
    pairs_new = list()
    for pair in pairs:
        if pair[0] != ',':
            pairs_new.append(pair)
    return pairs_new

In [3]:
fnames = list()
for subdir in sorted(os.listdir(path_data)):
    fnames.append(os.path.join(path_data, subdir))
    
gt_spks, gt_intents, gt_tags, gt_pairs, spokens = list(), list(), list(), list(), list()
path_txts_all = list()
for fname in fnames:
    path_audios, path_txts, gt_route = extract_data(os.path.join(path_data, fname))
    transcripts, spks, intents, tags, pairs = extract_NER(path_txts)
    spoken = normalize_transcripts(transcripts)
    gt_spks.extend(spks)
    gt_intents.extend(intents)
    gt_tags.extend(tags)
    gt_pairs.extend(pairs)
    spokens.extend(spoken)
    path_txts_all.extend(path_txts)
print(len(spokens))    

1413


In [4]:
pred_spks, pred_intents, pred_pairs, pred_tags  = list(), list(), list(), list()
for i in range(len(spokens)):
    text = manager.add_commas(spokens[i])
    pairs, intent, speaker, tag = roberta.predict(text)
    if intent in ['greet', 'standby']:
        intent = 'other'
    pred_spks.append(speaker)
    pred_intents.append(intent)
    pred_pairs.append(pairs)
    pred_tags.append(tag)
#     if speaker != gt_spks[i]:
#     if gt_intents[i] == 'pushback' and intent != 'pushback':
#         print(i, text,':', intent, speaker, '-----', gt_intents[i], gt_spks[i])

In [36]:
ii = [900]
for i in ii:
    lines = FileHandler.read_txt(path_txts_all[i])
    lines[1], lines[2] = 'pilot', 'readback'
#     lines[1], lines[2] = 'controller', 'standby'
#     lines[1], lines[2] = 'pilot', 'other'
    FileHandler.write_txt(path_txts_all[i], lines)

In [6]:
for i in range(len(spokens)):
    if pred_spks[i] != gt_spks[i]:
        print(i, manager.add_commas(spokens[i]),':', pred_intents[i], pred_spks[i], '-----', gt_intents[i], gt_spks[i])

16 one one eight decimal two five air france two five , good night : readback pilot ----- frequency controller
98 ground cathay six five seven , sierra , hold short romeo seven : other pilot ----- taxi controller
114 ground , good evening , china eastern five four five right turn whiskey : other controller ----- other pilot
122 china eastern five six six , singapore ground , good evening , i , copy , your request approved start one engine on idle power call me again when you are ready for pushback : other pilot ----- other controller
124 china eastern five six six , i say again approved start one engine on idle power : other pilot ----- other controller
197 singapore ground , good evening , fedex six zero eight one , heavy taxi with information foxtrot : other pilot ----- other controller
204 singapore ground , good evening , fedex six zero eight one , heavy on tango , hold short romeo one : other controller ----- other pilot
213 ok , fedex six zero eight one , is tango , to tango one 

In [40]:
key = 'taxi'
for i in range(len(spokens)):
    if gt_intents[i] == key and pred_intents[i] != key:
        print(i, manager.add_commas(spokens[i]),':', pred_intents[i], pred_spks[i], '-----', gt_intents[i], gt_spks[i])
    if gt_intents[i] != key and pred_intents[i] == key:
        print(i, manager.add_commas(spokens[i]),':', pred_intents[i], pred_spks[i], '-----', gt_intents[i], gt_spks[i])                

141 singapore one nine three , standby , further hold short romeo one , initially : other controller ----- taxi controller
188 singapore ground , good evening , singapore three five six , turn onto romeo , to hold short romeo seven : taxi controller ----- other pilot
249 singapore ground , hello , speedbird one six , taxi on romeo , hold short romeo seven : taxi controller ----- other pilot


In [42]:
result = compute_per_class_metrics(gt_spks, pred_spks)
print(result['f1_per_class'])
result = compute_per_class_metrics(gt_intents, pred_intents)
print(np.unique(gt_intents), result['f1_per_class'])

[0.96028881 0.96755162]
['frequency' 'other' 'pushback' 'readback' 'taxi' 'traffic'] [0.98360656 0.96103896 0.95652174 0.98617512 0.98013245 1.        ]


In [43]:
for i in range(len(gt_pairs)):
    gt_ner, pred_ner = remove_commas(gt_pairs[i]), remove_commas(pred_pairs[i])
    result = score_entities_per_label(gt_ner, pred_ner)
    result = result['per_label']
    if "TAXIWAY" in result.keys():
        if result['TAXIWAY']['f1'] < 0.95:
            print('----------', i, manager.add_commas(spokens[i]), '----------')
            
            
            print(pred_ner)
            print(gt_ner)


---------- 48 ground , good evening , scooter six one one , whiskey five left ----------
[('ground', 'CONTROLLER'), ('good evening', 'GREETING'), ('scooter six one one', 'CALLSIGN'), ('whiskey five left', 'GATE')]
[['ground', 'CONTROLLER'], ['good evening', 'GREETING'], ['scooter six one one', 'CALLSIGN'], ['whiskey five', 'TAXIWAY'], ['left', 'QUALIFIER']]
---------- 50 left whiskey , papa , quebec three , quebec two , delta , delta three zero , scooter six one one ----------
[('left', 'O'), ('whiskey', 'TAXIWAY'), ('papa', 'TAXIWAY'), ('quebec three', 'TAXIWAY'), ('quebec two', 'TAXIWAY'), ('delta', 'TAXIWAY'), ('delta three zero', 'GATE'), ('scooter six one one', 'CALLSIGN')]
[['left', 'QUALIFIER'], ['whiskey', 'TAXIWAY'], ['papa', 'TAXIWAY'], ['quebec three', 'TAXIWAY'], ['quebec two', 'TAXIWAY'], ['delta', 'O'], ['delta three zero', 'GATE'], ['scooter six one one', 'CALLSIGN']]
---------- 51 ground scooter six zero seven , whiskey five , quebec ----------
[('ground', 'CONTROLLER')

In [29]:
i = 70
lines = FileHandler.read_txt(path_txts_all[i])
lines

['ground , batik seven one five seven [UNK] whiskey five turn right on whiskey',
 'pilot',
 'other',
 'ground - CONTROLLER',
 ', - O',
 'batik seven one five seven - CALLSIGN',
 '[UNK] - O',
 'whiskey five - TAXIWAY',
 'turn - ACTION',
 'right - QUALIFIER',
 'on - O',
 'whiskey - TAXIWAY']

In [30]:
lines[0] = 'ground , batik seven one five seven vacate whiskey five turn right on whiskey'
lines[5] = 'vacate - ACTION'
# lines[8] = 'alpha nine - GATE'
# del lines[10]

In [31]:
lines

['ground , batik seven one five seven vacate whiskey five turn right on whiskey',
 'pilot',
 'other',
 'ground - CONTROLLER',
 ', - O',
 'vacate - ACTION',
 '[UNK] - O',
 'whiskey five - TAXIWAY',
 'turn - ACTION',
 'right - QUALIFIER',
 'on - O',
 'whiskey - TAXIWAY']

In [32]:
FileHandler.write_txt(path_txts_all[i], lines)

In [93]:
for i in range(len(spokens)):
    if gt_spks[i] == 'contoller':
        lines = FileHandler.read_txt(path_txts_all[i])
        lines[1] = 'controller'
        FileHandler.write_txt(path_txts_all[i], lines)

In [70]:
ii = [190]
for i in ii:
    lines = FileHandler.read_txt(path_txts_all[i])
    # lines[1] = 'pilot'
    lines[2] = 'taxi'
    FileHandler.write_txt(path_txts_all[i], lines)

In [106]:
for i in range(len(spokens)):
    lines = FileHandler.read_txt(path_txts_all[i])
    for j in range(3, len(lines)):
        cpn = lines[j].split(' - ')
        if cpn[0] == 'taxiway' and cpn[1] == 'TAXIWAY':
            lines[j] = 'taxiway - O'
    FileHandler.write_txt(path_txts_all[i], lines)

---------- 14 ----------
[['air france two five seven', 'CALLSIGN'], ['quebec three', 'TAXIWAY'], ['quebec', 'TAXIWAY'], ['tango', 'TAXIWAY'], ['holding point', 'ACTION'], ['tango one two', 'TAXIWAY']]
[('air france two five seven', 'CALLSIGN'), ('quebec three', 'TAXIWAY'), ('quebec', 'O'), ('tango', 'TAXIWAY'), ('holding point', 'ACTION'), ('tango one two', 'TAXIWAY')]
---------- 21 ----------
[['air india two one one six', 'CALLSIGN'], ['echo six', 'GATE'], ['pushback', 'ACTION'], ['approved', 'STATUS'], ['to', 'O'], ['face', 'ACTION'], ['south', 'QUALIFIER'], ['on', 'O'], ['taxiway', 'TAXIWAY'], ['tango', 'TAXIWAY']]
[('air india two one one six', 'CALLSIGN'), ('echo six', 'GATE'), ('pushback', 'O'), ('approved', 'STATUS'), ('to', 'O'), ('face', 'O'), ('south', 'QUALIFIER'), ('on', 'O'), ('taxiway', 'ACTION'), ('tango', 'TAXIWAY')]
---------- 22 ----------
[['pushback', 'ACTION'], ['approved', 'STATUS'], ['facing', 'ACTION'], ['south', 'QUALIFIER'], ['on', 'O'], ['taxiway', 'TAXIWAY

In [105]:
i = 14
gt_ner, pred_ner = remove_commas(gt_pairs[i]), remove_commas(pred_pairs[i])
print(gt_ner)
print(pred_ner)

[['air france two five seven', 'CALLSIGN'], ['quebec three', 'TAXIWAY'], ['quebec', 'TAXIWAY'], ['tango', 'TAXIWAY'], ['holding point', 'ACTION'], ['tango one two', 'TAXIWAY']]
[('air france two five seven', 'CALLSIGN'), ('quebec three', 'TAXIWAY'), ('quebec', 'O'), ('tango', 'TAXIWAY'), ('holding point', 'ACTION'), ('tango one two', 'TAXIWAY')]


In [100]:
result['per_label']

{'ACTION': {'precision': 0.9999999989999999,
  'recall': 0.9999999989999999,
  'f1': 0.9999999984999999,
  'support': 1,
  'TP': 1,
  'FP': 0,
  'FN': 0},
 'CALLSIGN': {'precision': 0.9999999989999999,
  'recall': 0.9999999989999999,
  'f1': 0.9999999984999999,
  'support': 1,
  'TP': 1,
  'FP': 0,
  'FN': 0},
 'CONTROLLER': {'precision': 0.9999999989999999,
  'recall': 0.9999999989999999,
  'f1': 0.9999999984999999,
  'support': 1,
  'TP': 1,
  'FP': 0,
  'FN': 0},
 'TAXIWAY': {'precision': 0.9999999989999999,
  'recall': 0.9999999989999999,
  'f1': 0.9999999984999999,
  'support': 1,
  'TP': 1,
  'FP': 0,
  'FN': 0}}

In [4]:
path_label = '/home/thaivanphat/OneDrive/WorkSpace/Data/Speech/Changi/VHF/Labels/ASR'
path_csv = os.path.join(path_label, 'train.csv')
path_audios, texts = AudioHandler.extract_csv(path_csv)

In [37]:
callsigns = list()
for i in range(len(texts)):
    text = normalize_text(texts[i])
    pairs, intent, speaker = model.predict(text)
    if intent != 'taxi':
        continue
    flag = 0
    for p in range(len(pairs)):
        if pairs[p][1] == 'CALLSIGN':
            callsigns.append(pairs[p][0])
            flag = 1
    if flag == 1:
        print('-----', i, intent, speaker, '-----')
        print(text)
        print(pairs)

callsigns = list(set(callsigns))    

----- 3 taxi controller -----
singapore one seven eight taxi tango holding point tango one two runway zero two center
[('singapore one seven eight', 'CALLSIGN'), ('taxi', 'ACTION'), ('tango', 'TAXIWAY'), ('holding point', 'ACTION'), ('tango one two', 'TAXIWAY'), ('runway zero two center', 'RUNWAY')]
----- 10 taxi controller -----
express india six niner zero singapore ground , good morning , taxi via romeo seven hold short romeo five
[('express india six niner zero', 'CALLSIGN'), ('singapore ground', 'CONTROLLER'), (',', 'O'), ('good morning', 'GREETING'), (',', 'O'), ('taxi', 'ACTION'), ('via', 'O'), ('romeo seven', 'TAXIWAY'), ('hold short', 'ACTION'), ('romeo five', 'TAXIWAY')]
----- 22 taxi controller -----
singapore four three two , negative , on tango hold short romeo
[('singapore four three two', 'CALLSIGN'), (',', 'O'), ('negative', 'ACTION'), (',', 'O'), ('on', 'O'), ('tango', 'TAXIWAY'), ('hold short', 'ACTION'), ('romeo', 'TAXIWAY')]
----- 25 taxi controller -----
[UNK] six 

----- 136 taxi controller -----
scooter two zero , taxi Quebec three , Quebec , tango , hold short romeo one
[('scooter two zero', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('quebec three', 'TAXIWAY'), (',', 'O'), ('quebec', 'TAXIWAY'), (',', 'O'), ('tango', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('romeo one', 'TAXIWAY')]
----- 137 taxi controller -----
Quebec three , Quebec , tango , hold short romeo one , scooter two zero
[('quebec three', 'TAXIWAY'), (',', 'O'), ('quebec', 'TAXIWAY'), (',', 'O'), ('tango', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('romeo one', 'TAXIWAY'), (',', 'O'), ('scooter two zero', 'CALLSIGN')]
----- 145 taxi controller -----
scooter five one zero , taxi papa four , hold short tango
[('scooter five one zero', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('papa four', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('tango', 'TAXIWAY')]
----- 148 taxi controller -----
scooter six four two , taxi papa , tango , hold short papa four
[('sc

----- 212 taxi controller -----
Indonesia eight three eight Singapore ground , good evening , turn right whiskey , hold short victor seven
[('indonesia eight three eight', 'CALLSIGN'), ('singapore ground', 'CONTROLLER'), (',', 'O'), ('good evening', 'GREETING'), (',', 'O'), ('turn', 'ACTION'), ('right', 'O'), ('whiskey', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('victor seven', 'TAXIWAY')]
----- 228 taxi controller -----
Indonesia eight three eight , taxi victor seven , victor , hold short victor one two
[('indonesia eight three eight', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('victor seven', 'TAXIWAY'), (',', 'O'), ('victor', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('victor one two', 'TAXIWAY')]
----- 231 taxi controller -----
china eastern five six five , singapore ground , turn right whiskey , hold short victor six
[('china eastern five six five', 'CALLSIGN'), (',', 'O'), ('singapore ground', 'CONTROLLER'), (',', 'O'), ('turn', 'ACTION'), ('right', 'O'), ('wh

----- 354 taxi controller -----
singapore one five four , taxi romeo seven , Romeo , hold short tango
[('singapore one five four', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('romeo seven', 'TAXIWAY'), (',', 'O'), ('romeo', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('tango', 'TAXIWAY')]
----- 359 taxi controller -----
cathay six five eight , taxi uniform , uniform one three , tango one three holding point
[('cathay six five eight', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('uniform', 'TAXIWAY'), (',', 'O'), ('uniform one three', 'TAXIWAY'), (',', 'O'), ('tango one three', 'TAXIWAY'), ('holding point', 'ACTION')]
----- 361 taxi controller -----
singapore seven four four zero , taxi tango , tango one two holding point , runway zero two center
[('singapore seven four four zero', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('tango', 'TAXIWAY'), (',', 'O'), ('tango one two', 'TAXIWAY'), ('holding point', 'ACTION'), (',', 'O'), ('runway zero two center', 'RUNWAY')]
----- 369 t

----- 441 taxi controller -----
Singapore seven four four zero , continue taxi papa , tango , hold short romeo one
[('singapore seven four four zero', 'CALLSIGN'), (',', 'O'), ('continue', 'ACTION'), ('taxi', 'ACTION'), ('papa', 'TAXIWAY'), (',', 'O'), ('tango', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('romeo one', 'TAXIWAY')]
----- 449 taxi controller -----
scooter six four two , taxi November , November four , papa , hold short tango
[('scooter six four two', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('november', 'TAXIWAY'), (',', 'O'), ('november four', 'TAXIWAY'), (',', 'O'), ('papa', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('tango', 'TAXIWAY')]
----- 455 taxi controller -----
scooter six four two , approaching tango ,
[('scooter six four two', 'CALLSIGN'), (',', 'O'), ('approaching', 'O'), ('tango', 'TAXIWAY'), (',', 'O')]
----- 456 taxi controller -----
scooter six four two , continue tango , hold short romeo one
[('scooter six four two', 'CALLSIGN'), (',',

----- 538 taxi controller -----
[UNK] , five four zero niner , Singapore ground , taxi romeo seven for bay foxtrot five niner
[('[UNK]', 'CALLSIGN'), (', five four zero niner', 'CALLSIGN'), (',', 'O'), ('singapore ground', 'CONTROLLER'), (',', 'O'), ('taxi', 'ACTION'), ('romeo seven', 'TAXIWAY'), ('for', 'O'), ('bay foxtrot five niner', 'GATE')]
----- 539 taxi controller -----
romeo seven for bay foxtrot five niner thai [UNK]
[('romeo seven', 'TAXIWAY'), ('for', 'O'), ('bay foxtrot five niner', 'GATE'), ('thai [UNK]', 'CALLSIGN')]
----- 540 taxi controller -----
qantas three six , taxi tango , holding point tango one two , runway zero two center
[('qantas three six', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('tango', 'TAXIWAY'), (',', 'O'), ('holding point', 'ACTION'), ('tango one two', 'TAXIWAY'), (',', 'O'), ('runway zero two center', 'RUNWAY')]
----- 542 taxi controller -----
bangkok air nine six two , taxi tango , holding point tango one two , runway zero two center
[('bangkok 

----- 656 taxi controller -----
qantas five two , taxi Quebec , tango , hold short romeo one
[('qantas five two', 'CALLSIGN'), (',', 'O'), ('taxi', 'ACTION'), ('quebec', 'TAXIWAY'), (',', 'O'), ('tango', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('romeo one', 'TAXIWAY')]
----- 658 taxi controller -----
qantas five two recleared turn right Quebec , hold short Quebec three
[('qantas five two', 'CALLSIGN'), ('recleared', 'TAXIWAY'), ('turn', 'ACTION'), ('right', 'O'), ('quebec', 'TAXIWAY'), (',', 'O'), ('hold short', 'ACTION'), ('quebec three', 'TAXIWAY')]
----- 663 taxi controller -----
scooter four six six , uh standby taxi sir
[('scooter four six six', 'CALLSIGN'), (',', 'O'), ('uh standby', 'TAXIWAY'), ('taxi', 'ACTION'), ('sir', 'O')]
----- 672 taxi controller -----
singapore three four taxi forward hold short tango
[('singapore three four', 'CALLSIGN'), ('taxi', 'ACTION'), ('forward', 'ACTION'), ('hold short', 'ACTION'), ('tango', 'TAXIWAY')]
----- 680 taxi controller -----


----- 795 taxi controller -----
singapore eight niner two go ahead
[('singapore eight niner two', 'CALLSIGN'), ('go', 'ACTION'), ('ahead', 'O')]
----- 798 taxi controller -----
singapore eight nine two recleared runway zero two left revise S I D  mersing six echo departure the rest of ATC clearance remain unchanged
[('singapore eight nine two', 'CALLSIGN'), ('recleared', 'TAXIWAY'), ('runway zero two left', 'RUNWAY'), ('rev', 'ACTION'), ('##ise', 'O'), ('s', 'O'), ('i', 'O'), ('d', 'I-TAXIWAY'), ('mersing', 'I-TAXIWAY'), ('six', 'O'), ('echo', 'O'), ('departure', 'O'), ('the', 'O'), ('rest', 'O'), ('of', 'O'), ('at', 'O'), ('##c', 'O'), ('clearance', 'O'), ('remain', 'O'), ('unchanged', 'O')]
----- 810 taxi controller -----
scooter six four two taxi via papa correction taxi via november papa eight hold short papa four
[('scooter six four two', 'CALLSIGN'), ('taxi', 'ACTION'), ('via', 'O'), ('papa', 'TAXIWAY'), ('correction', 'ACTION'), ('taxi', 'ACTION'), ('via', 'O'), ('november', 'TA

In [23]:
# callsign + monitor/contact + [singapore] tower/ground + freq
# special: negative, correction
# hold short, clear, taxi, continue
# left, right, ahead
# gate, bay, stand

In [ ]:
print('text: ', text)
print("Speaker:", speaker)
print("Intent:", intent)

In [2]:
def loadLabels(path_label):
    with open(path_label) as f:
        labels = json.load(f)
    return labels

def extractSpoken(labels):
    def _add_space(text):
        return re.sub(r'(\S)([,.])', r'\1 \2', text)
    spokens = list()
    for i in range(len(labels)):
        spoken = labels[i]['spoken'].lower()
        spoken = _add_space(spoken)
        spokens.append(spoken)
    return spokens

def predict_ner(sentence):
    """
    Tokenizes the sentence, makes predictions, and aligns back to words.
    """
    # Tokenize sentence with word alignment
    tokens = tokenizer(sentence, truncation=True, padding="max_length", max_length=64, return_tensors="pt",
                       return_offsets_mapping=True, is_split_into_words=True)
    
    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    offset_mapping = tokens["offset_mapping"].squeeze(0).tolist()
    word_ids = tokens.word_ids()  # Map token indices to words
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
    
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).squeeze(0).tolist()  # Convert logits to label IDs
    
    aligned_predictions = []
    seen_words = set()  # Track words we have already processed

    for i, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen_words:
            continue  # Skip special tokens and subwords
        
        label = id2label[predictions[i]]
        token = sentence[word_id]
#         token = tokenizer.convert_ids_to_tokens(input_ids[0][i].item())
        aligned_predictions.append((token, label))
        seen_words.add(word_id)  # Mark this word as processed

    return aligned_predictions

def merge_ner_results(ner_results):
    merged_results = []
    current_entity = []
    current_label = None

    for word, tag in ner_results:
        if tag.startswith("B-"):  # Start of a new entity
            if current_entity:  # Save the previous entity
                merged_results.append((" ".join(current_entity), current_label))
            current_entity = [word]  # Start a new entity
            current_label = tag[2:]  # Remove 'B-' prefix
        elif tag.startswith("I-") and current_label == tag[2:]:  # Inside same entity
            current_entity.append(word)
        else:  # Outside or new entity
            if current_entity:  # Save the previous entity
                merged_results.append((" ".join(current_entity), current_label))
                current_entity = []
                current_label = None
            merged_results.append((word, tag))  # Add non-entity words

    # Add last entity if any
    if current_entity:
        merged_results.append((" ".join(current_entity), current_label))

    return merged_results

def convertSpoken(word):
    num_dict = { "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "niner": 9, "ten": 10}
    alphabet_dict = {"alpha": "a", "bravo": "b", "charlie": "c", "delta": "d", "echo": "e", "foxtrot": "f", "golf": "g", "hotel": "h",  "india": "i", "juliett": "j",
                     "kilo": "k", "lima": "l", "mike": "m", "november": "n", "oscar": "o", "papa": "p", "quebec": "q", "romeo": "r", "sierra": "s", "tango": "t",
                     "uniform": "u", "victor": "v", "whiskey": "w", "xray": "x",  "yankee": "y", "zulu": "z"}
    runway_dict = {'left': 'l', 'center': 'c', 'right': 'l'}

    num, alphabet, runway = num_dict.get(word.lower(), word), alphabet_dict.get(word.lower(), word), runway_dict.get(word.lower(), word)
    if num != word:
        return str(num)
    if alphabet != word:
        return alphabet
    if runway != word:
        return runway
    return word

def convertEntity(words):
    words_new = list()
    for word in words:
        words_new.append(convertSpoken(word))
    return ''.join(words_new)

def predictRoute(spoken):
    words = spoken.split(' ')
    ner_results = predict_ner(words)
    merged = merge_ner_results(ner_results)
    instruction, destination = list(), list()
    for entity in merged:
        words = entity[0].split(' ')
        if entity[1] == 'Taxiway':
            instruction.append(convertEntity(words))
        if entity[1] in ['Gate', 'Runway']:
            destination.append(words[0])
            destination.append(convertEntity(words[1:]))
    return merged, instruction, destination

def parseInstruction(label):
    instruction = list()
    route = label['route']
    for r in route:
        instruction.append(r[1].lower())
    des = label['destination'].lower()
    destination = des.split(' ')
    return instruction, destination

In [11]:
ners, preds = list(), list()
for i in range(len(texts)):
    ner, ins, des = predictRoute(texts[i])
    ners.append(ner)
    preds.append([ins, des])


KeyboardInterrupt



In [3]:
path = '/home/thaivanphat/OneDrive/WorkSpace/Data/Speech/Changi/'
path_label = os.path.join(path, 'Labels/samples.json')
labels = loadLabels(path_label)
spokens = extractSpoken(labels)

In [9]:
ners, preds, gts = list(), list(), list()
for i in range(len(spokens)):
    ner, ins, des = predictRoute(spokens[i])
    ners.append(ner)
    preds.append([ins, des])
    gts.append(parseInstruction(labels[i]['parsed']['instructions']))

In [15]:
i = 0
print(spokens[i])
print(preds[i])

jetstar two zero six exit whiskey five , cleared to taxi to gate charlie two three , via whiskey , victor six , left on victor to victor one three
[['w5', 'w', 'v6', 'v', 'v13'], ['gate', 'c23']]


In [14]:
ners[i]

[('jetstar two zero six', 'Callsign'),
 ('exit', 'O'),
 ('whiskey five', 'Taxiway'),
 (',', 'O'),
 ('cleared', 'O'),
 ('to', 'O'),
 ('taxi', 'O'),
 ('to', 'O'),
 ('gate charlie two three', 'Gate'),
 (',', 'O'),
 ('via', 'O'),
 ('whiskey', 'Taxiway'),
 (',', 'O'),
 ('victor six', 'Taxiway'),
 (',', 'O'),
 ('left', 'Qualifier'),
 ('on', 'O'),
 ('victor', 'Taxiway'),
 ('to', 'O'),
 ('victor one three', 'Taxiway')]

In [63]:
for i in range(len(preds)):
    comp1 = preds[i][0] == gts[i][0]
    comp2 = preds[i][1] == gts[i][1]
    if comp1 + comp2 != 2:
        print(i, spokens[i])
        print(preds[i])
        print(gts[i])
        print('----')

In [66]:
words = spoken.split(' ')
ner_results = predict_ner(words)
merged = merge_ner_results(ner_results)


In [67]:
ner_results

[('jetstar', 'B-Callsign'),
 ('two', 'I-Callsign'),
 ('zero', 'I-Callsign'),
 ('six', 'I-Callsign'),
 ('exit', 'O'),
 ('whiskey', 'B-Taxiway'),
 ('five', 'I-Taxiway'),
 (',', 'O'),
 ('cleared', 'O'),
 ('to', 'O'),
 ('taxi', 'O'),
 ('to', 'O'),
 ('gat', 'B-Taxiway'),
 ('charlie', 'B-Taxiway'),
 ('too', 'I-Taxiway'),
 ('three', 'I-Taxiway'),
 (',', 'O'),
 ('via', 'O'),
 ('whiskey', 'B-Taxiway'),
 (',', 'O'),
 ('victor', 'B-Taxiway'),
 ('six', 'I-Taxiway'),
 (',', 'O'),
 ('left', 'B-Qualifier'),
 ('on', 'O'),
 ('victor', 'B-Taxiway'),
 ('to', 'O'),
 ('victor', 'B-Taxiway'),
 ('one', 'I-Taxiway'),
 ('three', 'I-Taxiway')]